# Blender One-Click V2 — ChatGPT Work

Versión limpia y robusta. **No usa `git clone`**, por lo que evita el error de directorio borrado que apareció en la versión anterior.

Uso: activa **GPU T4**, pulsa **Ejecutar todo**, autoriza Drive y espera. Antes del render completo hace un smoke test de 1 frame y cancela automáticamente si el backend no responde a tiempo.


In [ ]:
# @title Blender One-Click V2 — ejecutar todo
# Flujo robusto: no usa git clone, evita directorios de trabajo borrados,
# hace smoke test y solo entonces lanza el render completo.

PROJECT_SLUG = "blender-neon-core-demo"
REPO_ZIP = "https://github.com/izquierdodavid528-code/Chat-gpt-work-/archive/refs/heads/main.zip"
REPO_DIR = "/content/chatgpt-workspace"
BLENDER_VERSION = "4.5.14"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

import os
os.chdir("/content")

import json
import time
import shutil
import pathlib
import subprocess
import threading
import urllib.request
import zipfile
import tarfile
from datetime import datetime, timezone

def section(title):
    print("\n" + "="*76)
    print(title)
    print("="*76)

def run(cmd, check=True, capture=False):
    # Siempre ejecutar desde /content: nunca heredar un cwd borrado.
    p = subprocess.run(
        ["bash", "-lc", cmd],
        cwd="/content",
        text=True,
        capture_output=capture,
    )
    if capture:
        if p.stdout:
            print(p.stdout)
        if p.stderr:
            print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallo ({p.returncode}): {cmd}")
    return p

section("1/9 - Verificando GPU de Colab")
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
     "--format=csv,noheader"],
    cwd="/content",
    text=True,
    capture_output=True,
)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError(
        "No hay una GPU NVIDIA activa. En Colab selecciona "
        "Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > GPU T4, "
        "guarda y vuelve a Ejecutar todo."
    )
print("GPU:", gpu.stdout.strip())

section("2/9 - Montando Google Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/Remotion Projects")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive OK:", DRIVE_ROOT)

section("3/9 - Descargando workspace SIN git")
workspace_zip = pathlib.Path("/content/workspace-main.zip")
if pathlib.Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
if workspace_zip.exists():
    workspace_zip.unlink()

print("Descargando snapshot de GitHub...")
urllib.request.urlretrieve(REPO_ZIP, workspace_zip)

extract_root = pathlib.Path("/content/workspace-extract")
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)

with zipfile.ZipFile(workspace_zip, "r") as z:
    members = z.namelist()
    if not members:
        raise RuntimeError("El ZIP del repositorio llego vacio.")
    top_dir = members[0].split("/")[0]
    z.extractall(extract_root)

src_root = extract_root / top_dir
if not src_root.exists():
    raise RuntimeError(f"No se encontro la raiz extraida: {src_root}")
shutil.move(str(src_root), REPO_DIR)
shutil.rmtree(extract_root, ignore_errors=True)
workspace_zip.unlink(missing_ok=True)
os.chdir(REPO_DIR)
print("Workspace OK:", REPO_DIR)

section("4/9 - Instalando Blender 4.5.14 LTS + Vulkan")
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg libegl1 libgl1 libx11-6 libxi6 libxrender1 libxfixes3 libxxf86vm1 libxkbcommon0 libvulkan1 vulkan-tools")

archive = pathlib.Path("/content/blender.tar.xz")
if not pathlib.Path(BLENDER_BIN).exists():
    blender_url = f"https://download.blender.org/release/Blender4.5/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    print("Descargando Blender:", blender_url)
    urllib.request.urlretrieve(blender_url, archive)
    with tarfile.open(archive, "r:xz") as tf:
        tf.extractall("/content")
    archive.unlink(missing_ok=True)

ver = subprocess.run([BLENDER_BIN, "--version"], cwd="/content", text=True, capture_output=True)
print("\n".join(ver.stdout.splitlines()[:3]))
if ver.returncode != 0:
    raise RuntimeError("Blender no pudo iniciarse.")

vk = run("vulkaninfo --summary 2>/dev/null | head -n 100", check=False, capture=True)
if "NVIDIA" not in (vk.stdout or "") and "Tesla T4" not in (vk.stdout or ""):
    print("AVISO: Vulkan no identifico claramente NVIDIA. El smoke test decidira si continuar.")

section("5/9 - Preparando proyecto")
project_dir = pathlib.Path(REPO_DIR) / "projects" / PROJECT_SLUG
config_path = project_dir / "project.config.json"
scene_path = project_dir / "scene.py"

if not config_path.exists() or not scene_path.exists():
    raise FileNotFoundError("Falta project.config.json o scene.py en el proyecto.")

cfg = json.load(open(config_path, encoding="utf-8"))
out_dir = project_dir / "out"
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

print(json.dumps(cfg, indent=2, ensure_ascii=False))

source = scene_path.read_text(encoding="utf-8")

# No gastamos tiempo generando el poster durante el benchmark.
source_no_poster = source.replace(
    'bpy.ops.render.render(write_still=True)',
    'print("COLAB_V2: poster omitido durante benchmark")',
    1,
)

# Los temporales viven DENTRO del proyecto para que ROOT/OUT sigan apuntando
# al mismo sitio que scene.py.
full_scene = project_dir / "_colab_full.py"
smoke_scene = project_dir / "_colab_smoke.py"
full_scene.write_text(source_no_poster, encoding="utf-8")

needle = 'scene.frame_set(1)\nbpy.ops.render.render(animation=True)'
replacement = (
    'scene.render.resolution_percentage = 25\n'
    'scene.frame_end = 1\n'
    'scene.frame_set(1)\n'
    'bpy.ops.render.render(animation=True)'
)
if needle not in source_no_poster:
    raise RuntimeError("No se pudo preparar el smoke test: cambio inesperado en scene.py.")
smoke_scene.write_text(source_no_poster.replace(needle, replacement, 1), encoding="utf-8")

base_env = os.environ.copy()
base_env.pop("LIBGL_ALWAYS_SOFTWARE", None)
base_env.pop("MESA_LOADER_DRIVER_OVERRIDE", None)

def blender_cmd(scene_file):
    return [
        BLENDER_BIN,
        "--gpu-backend", "vulkan",
        "-b",
        "--python", str(pathlib.Path(REPO_DIR) / "scripts" / "blender" / "render-with-progress.py"),
        "--", str(scene_file),
    ]

def launch_render(scene_file, progress_file, log_file, timeout=None, show_progress=True):
    env = base_env.copy()
    env["BLENDER_PROJECT_SLUG"] = PROJECT_SLUG
    env["BLENDER_PROGRESS_FILE"] = str(progress_file)
    cmd = blender_cmd(scene_file)
    print("Comando:", " ".join(cmd))
    started = time.time()
    last = None
    with open(log_file, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            cmd,
            cwd="/content",
            stdout=log,
            stderr=subprocess.STDOUT,
            env=env,
            text=True,
        )
        while proc.poll() is None:
            elapsed = time.time() - started
            if timeout and elapsed > timeout:
                proc.terminate()
                try:
                    proc.wait(15)
                except Exception:
                    proc.kill()
                raise RuntimeError(
                    f"Smoke test supero {timeout}s. Se cancela para no perder tiempo."
                )
            if show_progress and progress_file.exists():
                try:
                    d = json.load(open(progress_file, encoding="utf-8"))
                    stamp = (d.get("percent"), d.get("current_frame"), d.get("eta_seconds"))
                    if stamp != last:
                        print(
                            f"Progreso: {d.get('percent')}% | "
                            f"frame {d.get('current_frame')}/{d.get('frame_end')} | "
                            f"ETA {d.get('eta_seconds')} s"
                        )
                        last = stamp
                except Exception:
                    pass
            time.sleep(3)

    elapsed = time.time() - started
    if proc.returncode != 0:
        tail = "\n".join(open(log_file, errors="ignore").read().splitlines()[-120:])
        print("\nULTIMAS LINEAS DEL LOG:\n" + tail)
        raise RuntimeError("Blender fallo; revisa el log mostrado arriba.")
    return elapsed

section("6/9 - Smoke test: 1 frame al 25%")
smoke_progress = out_dir / "smoke-progress.json"
smoke_log = out_dir / "smoke-render.log"
smoke_elapsed = launch_render(
    smoke_scene,
    smoke_progress,
    smoke_log,
    timeout=120,
    show_progress=False,
)
print(f"SMOKE TEST OK: {smoke_elapsed:.1f} segundos")

# Limpiar archivos de render producidos por el smoke sin borrar sus logs.
for p in list(out_dir.iterdir()):
    if p.name not in {"smoke-progress.json", "smoke-render.log"} and p.is_file():
        p.unlink()

section("7/9 - Render completo")
progress_file = out_dir / "render-progress.json"
render_log = out_dir / "colab-render.log"
gpu_log = out_dir / "gpu-usage.csv"

stop_poll = threading.Event()
def poll_gpu():
    with open(gpu_log, "w", encoding="utf-8") as f:
        f.write("timestamp,gpu_util_percent,memory_used_mb\n")
        while not stop_poll.is_set():
            p = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                cwd="/content",
                text=True,
                capture_output=True,
            )
            if p.returncode == 0 and p.stdout.strip():
                vals = p.stdout.strip().splitlines()[0]
                f.write(f"{datetime.now(timezone.utc).isoformat()},{vals}\n")
                f.flush()
            stop_poll.wait(5)

poll = threading.Thread(target=poll_gpu, daemon=True)
poll.start()
try:
    elapsed = launch_render(full_scene, progress_file, render_log, timeout=None, show_progress=True)
finally:
    stop_poll.set()
    poll.join(timeout=10)

print(f"RENDER COMPLETO: {elapsed/60:.2f} min ({elapsed:.1f} s)")

section("8/9 - Verificando y normalizando video")
expected_video = cfg.get("videoFile")
expected_path = pathlib.Path(REPO_DIR) / expected_video if expected_video else None

candidates = []
for pattern in ("*.mp4", "*.mkv", "*.mov", "*.webm", "*.avi"):
    candidates.extend(out_dir.glob(pattern))

if expected_path and not expected_path.exists() and candidates:
    source_video = max(candidates, key=lambda p: p.stat().st_size)
    tmp_mp4 = out_dir / (expected_path.name + ".tmp.mp4")
    print("Normalizando:", source_video.name, "->", expected_path.name)
    q = subprocess.run(
        ["ffmpeg", "-y", "-i", str(source_video), "-c", "copy", str(tmp_mp4)],
        cwd="/content",
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if q.returncode != 0:
        subprocess.run(
            ["ffmpeg", "-y", "-i", str(source_video),
             "-c:v", "libx264", "-preset", "veryfast", "-crf", "18",
             "-an", str(tmp_mp4)],
            cwd="/content",
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    tmp_mp4.replace(expected_path)

utils = []
if gpu_log.exists():
    for line in gpu_log.read_text(encoding="utf-8").splitlines()[1:]:
        try:
            utils.append(float(line.split(",")[1].strip()))
        except Exception:
            pass

summary = {
    "project": PROJECT_SLUG,
    "blender_version": BLENDER_VERSION,
    "backend": "vulkan",
    "gpu": gpu.stdout.strip(),
    "smoke_seconds": round(smoke_elapsed, 1),
    "elapsed_seconds": round(elapsed, 1),
    "elapsed_minutes": round(elapsed / 60, 2),
    "gpu_util_avg_percent": round(sum(utils)/len(utils), 1) if utils else None,
    "gpu_util_max_percent": round(max(utils), 1) if utils else None,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
(out_dir / "colab-benchmark.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)
print(json.dumps(summary, indent=2))

section("9/9 - Copiando a Google Drive")
drive_project = cfg.get("driveProjectDir", PROJECT_SLUG)
drive_out = DRIVE_ROOT / drive_project / "renders-colab-v2"
drive_out.mkdir(parents=True, exist_ok=True)

for f in out_dir.iterdir():
    if f.is_file():
        shutil.copy2(f, drive_out / f.name)

print("\nLISTO")
print("Destino:", drive_out)
for f in sorted(drive_out.iterdir()):
    if f.is_file():
        print(f" - {f.name}: {f.stat().st_size/1024/1024:.2f} MB")
